In [9]:
import random
from rdkit import Chem
from molpher.core import MolpherMol, MolpherAtom
from molpher.core.morphing.operators import MorphingOperator
from rdkit.Chem.EnumerateStereoisomers import EnumerateStereoisomers, StereoEnumerationOptions
from rdkit.Chem import rdChemReactions
from rdkit.Chem import rdmolops
from rdkit.Chem import Descriptors  
from molpher.core import ExplorationTree as ETree

class SerThrGlycosylationGalNAc(MorphingOperator):
    """O-Glycosylation (mucin-type) σε Ser/Thr με α-D-GalNAc (στερεοχημικά ορθό)."""
    def __init__(self):
        super(SerThrGlycosylationGalNAc, self).__init__()
        self._name = "O-Glycosylation (Ser/Thr, alpha-D-GalNAc - Mucin type)"
        self._target_oxygens = []

        self.SERINE_PATTERN = Chem.MolFromSmarts(
            "[NX3][CX4H]([CH2][OX2H])[CX3](=O)[NX3,OX2,OX1-]"
        )
        self.THREONINE_PATTERN = Chem.MolFromSmarts(
            "[NX3][CX4H]([CX4H]([OX2H])[CH3])[CX3](=O)[NX3,OX2,OX1-]"
        )

        # Template: α-D-GalNAc, πλήρης στερεοχημεία
        # C1(α, axial OH) - C2(NHAc, equatorial) - C3(OH) - C4(OH, axial - GAL επιφάνεια) - C5(CH2OH)
        # SMILES κανονικού α-D-GalNAc (πηγή: PubChem CID 12810074 / ChEBI)
        self.GALNAC_TEMPLATE = Chem.MolFromSmiles(
            "OC[C@H]1O[C@H](O)[C@H](NC(C)=O)[C@@H](O)[C@H]1O"
        )

    def setOriginal(self, mol):
        super(SerThrGlycosylationGalNAc, self).setOriginal(mol)
        self._target_oxygens = []
        if not self.original: return
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: return

        for pattern in [self.SERINE_PATTERN, self.THREONINE_PATTERN]:
            if pattern is None: continue
            for match in rdkit_mol.GetSubstructMatches(pattern):
                oh_idx = self._find_oh_in_match(rdkit_mol, match)
                if oh_idx is not None and oh_idx not in self._target_oxygens:
                    self._target_oxygens.append(oh_idx)

    def _find_oh_in_match(self, rdkit_mol, match):
        for idx in match:
            atom = rdkit_mol.GetAtomWithIdx(idx)
            if atom.GetAtomicNum() == 8 and atom.GetTotalNumHs() == 1:
                return idx
        return None

    def _find_c1_and_oh(self, template_mol):
        """
        Βρίσκει δυναμικά τον ανωμερικό C1 (τον άνθρακα συνδεδεμένο με 2 οξυγόνα:
        το ενδοκυκλικό O-δακτυλίου και το εξωκυκλικό -OH) και το -OH του.
        Αυτό αποφεύγει εξάρτηση από σταθερό index στο SMILES string.
        """
        ring_info = template_mol.GetRingInfo()
        for atom in template_mol.GetAtoms():
            if atom.GetAtomicNum() != 6:
                continue
            if not ring_info.IsAtomInRingOfSize(atom.GetIdx(), 6):
                continue
            o_neighbors = [n for n in atom.GetNeighbors() if n.GetAtomicNum() == 8]
            ring_o = [n for n in o_neighbors if ring_info.IsAtomInRingOfSize(n.GetIdx(), 6)]
            exo_oh = [n for n in o_neighbors if n.GetTotalNumHs() == 1 and n.GetIdx() not in [x.GetIdx() for x in ring_o]]
            if len(ring_o) == 1 and len(exo_oh) == 1:
                return atom.GetIdx(), exo_oh[0].GetIdx()
        return None, None

    def morph(self):
        if not self.original or not self._target_oxygens:
            return MolpherMol(other=self.original.asRDMol())

        rdkit_mol = self.original.asRDMol()
        try:
            target_o_idx = random.choice(self._target_oxygens)

            n_atoms_before = rdkit_mol.GetNumAtoms()
            c1_local, oh_local = self._find_c1_and_oh(self.GALNAC_TEMPLATE)
            if c1_local is None:
                return MolpherMol(other=rdkit_mol)

            combined = Chem.CombineMols(rdkit_mol, self.GALNAC_TEMPLATE)
            rw_combined = Chem.RWMol(combined)

            c1_idx = n_atoms_before + c1_local
            oh_idx = n_atoms_before + oh_local

            # Γλυκοσιδικός δεσμός: O(Ser/Thr) - C1(γλυκάνη)
            rw_combined.AddBond(target_o_idx, c1_idx, Chem.BondType.SINGLE)
            rw_combined.RemoveAtom(oh_idx)

            # Αναπροσαρμογή index λόγω RemoveAtom
            c1_idx_adj = c1_idx - 1 if oh_idx < c1_idx else c1_idx

            new_mol = rw_combined.GetMol()
            for idx in [target_o_idx, c1_idx_adj]:
                atom = new_mol.GetAtomWithIdx(idx)
                atom.SetNoImplicit(False)
                atom.SetNumExplicitHs(0)
                atom.SetFormalCharge(0)

            new_mol.UpdatePropertyCache(strict=False)
            Chem.SanitizeMol(new_mol, Chem.SanitizeFlags.SANITIZE_ALL)
            Chem.AssignStereochemistry(new_mol, cleanIt=True, force=True)
            return MolpherMol(other=new_mol)

        except Exception as e:
            print(f"[Debug Error]: {e}")
            return MolpherMol(other=rdkit_mol)

    def getName(self): return self._name

print("=== TEST 1: Ala-Ser-GalNAc ===")
dipeptide_ser = MolpherMol("CC(N)C(=O)NC(CO)C(=O)O")  # Ala-Ser
op1 = SerThrGlycosylationGalNAc()
op1.setOriginal(dipeptide_ser)
print(f"Sites found: {len(op1._target_oxygens)}")
result1 = op1.morph()
print(f"Result: {result1.getSMILES()}\n")

print("=== TEST 2: Ser-Galnac ===")
serine = MolpherMol("NC(CO)C(=O)O")
op1.setOriginal(serine)
print(f"Sites found: {len(op1._target_oxygens)}")
result1 = op1.morph()
print(f"Result: {result1.getSMILES()}\n")

print("=== TEST: Thr-GalNAc ===")
threonine = MolpherMol("NC(C(O)C)C(=O)O")  # Thr
op1 = SerThrGlycosylationGalNAc()
op1.setOriginal(threonine)
print(f"Sites found: {len(op1._target_oxygens)}")
result1 = op1.morph()
print(f"Result: {result1.getSMILES()}\n")

=== TEST 1: Ala-Ser-GalNAc ===
Sites found: 1
Result: CC(=O)NC1C(OCC(NC(=O)C(C)N)C(=O)O)OC(CO)C(O)C1O

=== TEST 2: Ser-Galnac ===
Sites found: 1
Result: CC(=O)NC1C(OCC(N)C(=O)O)OC(CO)C(O)C1O

=== TEST: Thr-GalNAc ===
Sites found: 1
Result: CC(=O)NC1C(OC(C)C(N)C(=O)O)OC(CO)C(O)C1O



In [1]:
import random
from rdkit import Chem
from molpher.core import MolpherMol, MolpherAtom
from molpher.core.morphing.operators import MorphingOperator
from rdkit.Chem.EnumerateStereoisomers import EnumerateStereoisomers, StereoEnumerationOptions
from rdkit.Chem import rdChemReactions
from rdkit.Chem import rdmolops
from rdkit.Chem import Descriptors  
from molpher.core import ExplorationTree as ETree

class AsnGlycosylationGlcNAc(MorphingOperator):
    """N-Glycosylation σε Ασπαραγίνη με β-D-GlcNAc (στερεοχημικά ορθό, κανονικό μοντέλο)."""
    def __init__(self):
        super(AsnGlycosylationGlcNAc, self).__init__()
        self._name = "N-Glycosylation (Asparagine, beta-D-GlcNAc - canonical)"
        self._target_nitrogens = []
        self.ASN_PATTERN = Chem.MolFromSmarts(
            "[$([NX3H2]),$([NX3H1][#6])][CX4H]([CH2][CX3](=O)[NX3H2])[CX3](=O)[$([OX2H1]),$([OX1-]),$([NX3])]"
        )

        self.GLCNAC_TEMPLATE = Chem.MolFromSmiles(
            "OC[C@H]1O[C@H](O)[C@H](NC(C)=O)[C@H](O)[C@H]1O"
        )

    def setOriginal(self, mol):
        super(AsnGlycosylationGlcNAc, self).setOriginal(mol)
        self._target_nitrogens = []
        if not self.original: return
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: return

        if self.ASN_PATTERN is not None:
            for match in rdkit_mol.GetSubstructMatches(self.ASN_PATTERN):
                n_idx = self._find_amide_nh2(rdkit_mol, match)
                if n_idx is not None and n_idx not in self._target_nitrogens:
                    self._target_nitrogens.append(n_idx)

    def _find_amide_nh2(self, rdkit_mol, match):
        """
        Εντοπίζει το άζωτο της πλευρικής ομάδας εξετάζοντας το χημικό περιβάλλον
        των ατόμων του match, αποφεύγοντας τα σφάλματα σειράς του SMARTS.
        """
        for idx in match:
            atom = rdkit_mol.GetAtomWithIdx(idx)
            
            # Ψάχνουμε άζωτο (7) με 2 υδρογόνα (το Asn amide -NH2)
            if atom.GetAtomicNum() == 7 and atom.GetTotalNumHs() == 2:
                
                # Έλεγχος αν είναι συνδεδεμένο με το καρβονύλιο της πλευρικής αλυσίδας
                for neighbor in atom.GetNeighbors():
                    if neighbor.GetAtomicNum() == 6: # Καρβονύλιο (C=O)
                        
                        # Επιβεβαίωση ότι είναι όντως καρβονύλιο και συνδέεται με CH2 (C_beta)
                        has_double_bound_oxygen = False
                        attached_to_c_beta = False
                        
                        for bond in neighbor.GetBonds():
                            other = bond.GetOtherAtom(neighbor)
                            # 1. Έλεγχος για =O
                            if other.GetAtomicNum() == 8 and bond.GetBondType() == Chem.BondType.DOUBLE:
                                has_double_bound_oxygen = True
                            # 2. Έλεγχος αν ο άλλος άνθρακας είναι ο C_beta (έχει υδρογόνα και 4 απλούς δεσμούς)
                            elif other.GetAtomicNum() == 6 and other.GetIdx() in match:
                                # Ο C_alpha συνδέεται με καρβονύλιο πεπτιδικού δεσμού, 
                                # ενώ ο C_beta της πλευρικής αλυσίδας συνδέεται με το CH2
                                attached_to_c_beta = True
                                
                        if has_double_bound_oxygen and attached_to_c_beta:
                            # Βρήκαμε το σωστό άζωτο της πλευρικής ομάδας!
                            return idx
        return None

    def _find_c1_and_oh(self, template_mol):
        ring_info = template_mol.GetRingInfo()
        for atom in template_mol.GetAtoms():
            if atom.GetAtomicNum() != 6:
                continue
            if not ring_info.IsAtomInRingOfSize(atom.GetIdx(), 6):
                continue
            o_neighbors = [n for n in atom.GetNeighbors() if n.GetAtomicNum() == 8]
            ring_o = [n for n in o_neighbors if ring_info.IsAtomInRingOfSize(n.GetIdx(), 6)]
            exo_oh = [n for n in o_neighbors if n.GetTotalNumHs() == 1
                      and n.GetIdx() not in [x.GetIdx() for x in ring_o]]
            if len(ring_o) == 1 and len(exo_oh) == 1:
                return atom.GetIdx(), exo_oh[0].GetIdx()
        return None, None

    def morph(self):
        if not self.original or not self._target_nitrogens:
            return MolpherMol(other=self.original.asRDMol())

        rdkit_mol = self.original.asRDMol()
        try:
            target_n_idx = random.choice(self._target_nitrogens)

            n_atoms_before = rdkit_mol.GetNumAtoms()
            c1_local, oh_local = self._find_c1_and_oh(self.GLCNAC_TEMPLATE)
            if c1_local is None:
                return MolpherMol(other=rdkit_mol)

            combined = Chem.CombineMols(rdkit_mol, self.GLCNAC_TEMPLATE)
            rw_combined = Chem.RWMol(combined)

            c1_idx = n_atoms_before + c1_local
            oh_idx = n_atoms_before + oh_local

            rw_combined.AddBond(target_n_idx, c1_idx, Chem.BondType.SINGLE)
            rw_combined.RemoveAtom(oh_idx)

            c1_idx_adj = c1_idx - 1 if oh_idx < c1_idx else c1_idx

            new_mol = rw_combined.GetMol()
            
            for idx in [target_n_idx, c1_idx_adj]:
                atom = new_mol.GetAtomWithIdx(idx)
                atom.SetNoImplicit(False)
                atom.SetNumExplicitHs(0)
                atom.SetNumRadicalElectrons(0) # Κρίσιμο για να μην κρατάει ελεύθερα ριζικά ηλεκτρόνια
                atom.SetFormalCharge(0)

            new_mol.UpdatePropertyCache(strict=False)
            Chem.SanitizeMol(new_mol, Chem.SanitizeFlags.SANITIZE_ALL ^ Chem.SanitizeFlags.SANITIZE_KEKULIZE)
            Chem.Kekulize(new_mol, clearAromaticFlags=True)
            Chem.SanitizeMol(new_mol, Chem.SanitizeFlags.SANITIZE_KEKULIZE)
            
            Chem.AssignStereochemistry(new_mol, cleanIt=True, force=True)
            return MolpherMol(other=new_mol)

        except Exception as e:
            print(f"[Debug Error]: {e}")
            return MolpherMol(other=rdkit_mol)

    def getName(self): return self._name


print("=== TEST 1: Asn-Glucose (sequon Asn-Ala-Ser) ===")
tripeptide_asn = MolpherMol("NC(CC(N)=O)C(=O)NC(C)C(=O)NC(CO)C(=O)O")  # Asn-Ala-Ser
op3 = AsnGlycosylationGlcNAc()
op3.setOriginal(tripeptide_asn)
print(f"Sites found: {len(op3._target_nitrogens)}")
result2 = op3.morph()
print(f"Result: {result2.getSMILES()}\n")

print("=== TEST 2: Asn-GlcNAc (sequon Asn-Ala-Thr) ===")
tripeptide_asn2 = MolpherMol("NC(CC(N)=O)C(=O)NC(C)C(=O)NC(C(O)C)C(=O)O")  # Asn-Ala-Thr  # Asn-Ala-Thr
op3 = AsnGlycosylationGlcNAc()
op3.setOriginal(tripeptide_asn2)
print(f"Sites found: {len(op3._target_nitrogens)}")
result3 = op3.morph()
print(f"Result: {result3.getSMILES()}\n")

print("=== TEST 3: Free Asparagine ===")
asn = MolpherMol("NC(CC(N)=O)C(=O)O")
op4 = AsnGlycosylationGlcNAc()
op4.setOriginal(asn)
print(f"Sites found: {len(op4._target_nitrogens)}")
result4 = op4.morph()
print(f"Result: {result4.getSMILES()}\n")

print("=== TEST 4: Tripeptide Ala-Asn-Gly ===")
tripeptide = MolpherMol("CC(N)C(=O)NC(CC(N)=O)C(=O)NCC(=O)O")
op5 = AsnGlycosylationGlcNAc()
op5.setOriginal(tripeptide)
print(f"Sites found: {len(op5._target_nitrogens)}")
result5 = op5.morph()
print(f"Result: {result5.getSMILES()}\n")

=== TEST 1: Asn-Glucose (sequon Asn-Ala-Ser) ===
Sites found: 1
Result: CC(=O)NC1C(NC(=O)CC(N)C(=O)NC(C)C(=O)NC(CO)C(=O)O)OC(CO)C(O)C1O

=== TEST 2: Asn-GlcNAc (sequon Asn-Ala-Thr) ===
Sites found: 1
Result: CC(=O)NC1C(NC(=O)CC(N)C(=O)NC(C)C(=O)NC(C(=O)O)C(C)O)OC(CO)C(O)C1O

=== TEST 3: Free Asparagine ===
Sites found: 1
Result: CC(=O)NC1C(NC(=O)CC(N)C(=O)O)OC(CO)C(O)C1O

=== TEST 4: Tripeptide Ala-Asn-Gly ===
Sites found: 1
Result: CC(=O)NC1C(NC(=O)CC(NC(=O)C(C)N)C(=O)NCC(=O)O)OC(CO)C(O)C1O

